# Bank Marketing Exploratory Data Analysis
This notebook performs a comprehensive EDA on the Bank Marketing dataset to understand customer behavior and campaign effectiveness.

In [3]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

In [4]:
dataset_url = 'https://raw.githubusercontent.com/nawaz0x1/Quantropy/refs/heads/main/Exploratory%20Data%20Analysis/data/bank-marketing.csv'

df = pd.read_csv(dataset_url)

In [5]:
df.shape

(11162, 17)

In [6]:
df.head(30)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes
5,42,management,single,tertiary,no,0,yes,yes,unknown,5,may,562,2,-1,0,unknown,yes
6,56,management,married,tertiary,no,830,yes,yes,unknown,6,may,1201,1,-1,0,unknown,yes
7,60,retired,divorced,secondary,no,545,yes,no,unknown,6,may,1030,1,-1,0,unknown,yes
8,37,technician,married,secondary,no,1,yes,no,unknown,6,may,608,1,-1,0,unknown,yes
9,28,services,single,secondary,no,5090,yes,no,unknown,6,may,1297,3,-1,0,unknown,yes


In [8]:
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
if 'month' in cat_cols:
    cat_cols.remove('month')

for col in cat_cols:
    fig = px.histogram(df, x=col, color='deposit', barmode='group',
                       title=f'{col.capitalize()} by Deposit Outcome',
                       category_orders={col: df[col].value_counts().index.tolist()},
                       color_discrete_sequence=px.colors.qualitative.Safe)
    fig.update_layout(xaxis_title='', yaxis_title='Count', template='plotly_dark')
    fig.show()

C:\Users\milon\AppData\Local\Temp\ipykernel_10496\1605522594.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns.tolist()


## Behavioral Density: Engagement vs. Demographics
**Question:** At what call duration and age threshold does the probability of a deposit subscription peak across different marital statuses?

In [10]:
fig = px.density_contour(df, x='duration' , y='age', color='deposit',
                        facet_col='marital',
                        marginal_x='violin' ,
                        marginal_y='histogram',
                        title='Interactive Density Contour : Age , Duration, and Marital Success',
                        labels = {'duration' : 'Call Duration (sec)', 'age' : 'Age'})
fig.update_layout(template = 'plotly_dark',height=600)
fig.show()

## Financial Resilience: Education and Debt Influence
**Question:** How do education levels and housing loans interact to influence account balances and final conversion outcomes?

In [12]:
fig = px.box(df , x='education', y='balance', color='housing',
             notched = True,
             title='Interactive Financial Resilience : Balance by Education and Debt',
             labels= {'balance' : 'Balance' ,'education' : 'Education Level'})
fig.update_layout(yaxis_type ='log' , template ='plotly_dark')
fig.show()

## Temporal ROI: Monthly Campaign Efficiency
**Question:** Which months provide the highest conversion 'return on investment' despite having lower total contact volumes?

In [13]:
month_order = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
monthly_data = df.groupby(['month', 'deposit']).size().unstack().reindex(month_order)
monthly_ratio = monthly_data.div(monthly_data.sum(axis=1), axis=0)

fig = go.Figure()
fig.add_trace(go.Scatter(x=month_order, y=monthly_ratio['yes'], name='Conversion Rate', mode='lines+markers', yaxis='y2', line=dict(color='gold', width=4)))
fig.add_trace(go.Bar(x=month_order, y=monthly_data.sum(axis=1), name='Total Contacts', opacity=0.6, marker_color='teal'))

fig.update_layout(
    title='Monthly Campaign Volume vs. Conversion Efficiency',
    yaxis=dict(title='Total Contact Volume'),
    yaxis2=dict(title='Conversion Rate', overlaying='y', side='right'),
    legend=dict(x=1.1, y=1)
)
fig.show()

## Visualizing Campaign Success Thresholds
**Question:** Can we pinpoint the exact number of contacts where the probability of conversion begins to diminish significantly?

In [14]:
campaign_counts = df.groupby(['campaign', 'deposit']).size().unstack(fill_value=0)
campaign_rates = campaign_counts.div(campaign_counts.sum(axis=1), axis=0)['yes'] * 100

fig = px.area(campaign_rates.reset_index()[:15], x='campaign', y='yes',
              title='The Fatigue Threshold: Conversion Rate by Contact Frequency',
              labels={'campaign': 'Number of Contacts', 'yes': 'Success Rate (%)'})
fig.update_traces(line_color='firebrick')
fig.update_layout(template='plotly_white')
fig.show()

## Multi-Channel Conversion Architecture
**Question:** How does the choice of communication channel influence the relationship between engagement time and successful conversions?

In [15]:
fig = px.box(df, x='contact' , y='duration' , color ='deposit',
             points='outliers' ,notched=True,
             title = ' Channel Effectiveness: Duration Distribution by Contact Method',
             labels={'duration':'Call Duration (sec)', 'contact':'Contact Method'})
fig.update_layout (yaxis_type='log', template ='plotly_dark')
fig.show()

## Socioeconomic Profiling: Education vs. Job Roles
**Question:** Does higher education consistently lead to higher account balances across all job sectors, and how does this affect deposit propensity?

In [19]:
tree_df = (
    df.groupby(['education', 'job', 'deposit'])['balance']
      .median()
      .reset_index()
)


tree_df['color_balance'] = tree_df['balance'].clip(lower=0)

fig = px.treemap(
    tree_df,
    path=['education', 'job', 'deposit'],
    values=tree_df['balance'].abs() + 1,
    color='color_balance',
    color_continuous_scale='Viridis',
    range_color=(0, tree_df['balance'].clip(lower=0).max()),
    title='Hierarchical View: Education & Job vs. Median Balance'
)

fig.update_coloraxes(
    colorbar_title='Median Positive Balance'
)

fig.update_layout(
    margin=dict(t=50, l=25, r=25, b=25)
)

fig.show()

## The Interaction of Housing Debt and Marital Status
**Question:** Does marital status influence the likelihood of a deposit subscription differently for those burdened by housing loans versus those who are debt-free?

In [20]:
debt_marital = df.groupby(['housing', 'marital', 'deposit']).size().unstack()
debt_marital_pct = debt_marital.div(debt_marital.sum(axis=1), axis=0) * 100

fig = px.bar(debt_marital_pct.reset_index(), x='marital', y='yes', color='housing',
             barmode='group',
             title='Deposit Conversion Rate (%) by Marital Status and Housing Loan',
             labels={'yes': 'Conversion Rate (%)', 'marital': 'Marital Status', 'housing': 'Has Housing Loan'},
             color_discrete_sequence=px.colors.qualitative.Prism)
fig.show()

## Debt Portfolio & Subscription Propensity
**Question:** To what extent does the combination of housing and personal debt create a barrier to term deposit subscriptions?

In [21]:
debt_matrix = df.groupby(['housing', 'loan', 'deposit']).size().unstack(fill_value=0)
debt_matrix_pct = debt_matrix.div(debt_matrix.sum(axis=1), axis=0).reset_index()

fig = px.sunburst(debt_matrix_pct, path=['housing', 'loan'], values='yes',
                  color='yes', color_continuous_scale='RdYlGn',
                  title='The Debt Barrier: Subscription Rates by Loan Portfolio',
                  labels={'yes': 'Success Rate (%)', 'housing': 'Housing Loan', 'loan': 'Personal Loan'})
fig.show()

## Financial Stability Across the Life Cycle
**Question:** How does the relationship between age and account balance shift across different marital statuses, and what does this reveal about the bank's most 'liquid' segments?

In [25]:
fig = px.scatter(df, x='age', y='balance', color='marital', facet_col='marital',
                 trendline='lowess',
                 title='Interactive Life Cycle Analysis: Age vs. Balance Trends',
                 labels={'age': 'Age', 'balance': 'Account Balance'},
                 opacity=0.4)
fig.update_layout(yaxis_type='log', height=500, template='plotly_white')
fig.show()

## The Golden Window: Recency and Previous Outcomes
**Question:** How does the time elapsed since the last contact (pdays) interact with the success of previous campaigns to influence current deposit conversion?

In [26]:
recency_df = df[df['pdays'] != -1]

fig = px.scatter(recency_df, x='pdays', y='duration', color='deposit',
                 facet_col='poutcome',
                 marginal_x='box',
                 title='Conversion Dynamics: Recency (pdays) vs. Call Duration by Past Outcome',
                 labels={'pdays': 'Days Since Last Contact', 'duration': 'Duration (sec)'},
                 color_discrete_map={'yes': '#2ECC71', 'no': '#E74C3C'},
                 opacity=0.5)

fig.update_layout(template='plotly_white', height=500)
fig.show()

## Segment Profitability: Professional Roles vs. Personal Debt
**Question:** How does the presence of a personal loan affect the conversion rate across different job sectors? Which professions are most likely to subscribe despite carrying debt?

In [27]:
job_loan_counts = df.groupby(['job', 'loan', 'deposit']).size().unstack(fill_value=0)
job_loan_total = job_loan_counts.sum(axis=1)

job_loan_rate = (job_loan_counts['yes'] / job_loan_total * 100).reset_index(name='conv_rate')

job_loan_rate = job_loan_rate[job_loan_total.values > 0].copy()
job_loan_rate['plot_value'] = job_loan_rate['conv_rate'].apply(lambda x: x if x > 0 else 0.01)

fig = px.sunburst(job_loan_rate,
                  path=['job', 'loan'],
                  values='plot_value',
                  color='conv_rate',
                  color_continuous_scale='Portland',
                  title='Sunburst: Conversion Rates by Job and Personal Loan Status',
                  labels={'conv_rate': 'Actual Conv %', 'plot_value': 'Weight', 'loan': 'Has Personal Loan'})

fig.update_layout(height=700)
fig.show()

## Intra-Month Conversion Cycles
**Question:** Is there a specific period within the month (beginning, middle, or end) where customers are more likely to commit to a term deposit?

In [28]:
day_stats = df.groupby('day')['deposit'].value_counts(normalize=True).unstack().fillna(0).reset_index()

fig = px.line(day_stats, x='day', y='yes', markers=True,
              title='Interactive Intra-Month Conversion Probability',
              labels={'day': 'Day of Month', 'yes': 'Success Probability'})
fig.update_traces(line_color='indigo', fill='tozeroy')
fig.update_layout(xaxis=dict(tickmode='linear', tick0=1, dtick=1), template='plotly_white')
fig.show()

## Multivariate Success Pathways: Tracing the Ideal Customer
**Question:** When we combine education, housing debt, and personal loans, what specific path leads to the highest concentration of successful subscriptions?

In [29]:
path_df = df[['education', 'housing', 'loan', 'deposit']].copy()
path_df['deposit_numeric'] = path_df['deposit'].map({'yes': 1, 'no': 0})

fig = px.parallel_categories(path_df,
                             dimensions=['education', 'housing', 'loan', 'deposit'],
                             color="deposit_numeric",
                             color_continuous_scale=['#EF553B', '#00CC96'],
                             labels={'education': 'Education', 'housing': 'Housing Debt', 'loan': 'Personal Loan', 'deposit': 'Converted?'},
                             title="High-Dimensional Flow: Socio-Financial Paths to Subscription")

fig.update_layout(margin=dict(l=150, r=100, t=100, b=50), coloraxis_showscale=False)
fig.show()

## Balance Volatility & Conversion Probabilities
**Question:** Is there a 'sweet spot' for account balance? Does having a very high balance significantly increase conversion, or is there a point of diminishing returns?

In [31]:
df['balance_decile'] = pd.qcut(df['balance'], 10, labels=False, duplicates='drop')
decile_stats = df.groupby('balance_decile')['deposit'].value_counts(normalize=True).unstack().fillna(0).reset_index()

fig = px.bar(decile_stats, x='balance_decile', y='yes',
             title='Conversion Probability by Balance Decile (Low to High Wealth)',
             labels={'balance_decile': 'Balance Decile (0=Lowest, 9=Highest)', 'yes': 'Conversion Rate'},
             color='yes',
             color_continuous_scale='Tealgrn')

fig.update_layout(template='plotly_dark')
fig.show()

## Final Strategic Synthesis & Executive Recommendations

Following this deep-dive exploratory analysis of the bank marketing dataset, we have identified several 'best-in-class' indicators for future campaign optimization:

### 1. The Efficiency Frontier
*   **Duration vs. Frequency:** While call durations of 300-600 seconds significantly boost conversion, there is a clear 'Campaign Fatigue' threshold. After 4 contacts, conversion rates drop, and even longer calls fail to compensate for over-contacting.
*   **Recommendation:** Limit outreach to a maximum of 3-4 high-quality contacts per customer.

### 2. Temporal & Seasonal Goldmines
*   **High-ROI Windows:** March, September, October, and December are 'efficiency peaks' where conversion rates exceed 80%.
*   **Recommendation:** Shift larger portions of the marketing budget to these high-performing shoulder months away from the high-volume/low-conversion month of May.

### 3. Professional & Educational Archetypes
*   **The Receptive Persona:** Students across all education levels and retired individuals show the highest baseline receptivity. Among working professionals, tertiary-educated administrators and technicians with no housing debt represent the 'liquid' goldmine.
*   **Recommendation:** Tailor messaging specifically for the 'student' and 'retired' segments as they show resilience to typical campaign fatigue.

### 4. Behavioral & Channel Indicators
*   **Channel Strategy:** Cellular contact yields higher conversion rates and facilitates longer, more meaningful engagements compared to landline telephones.
*   **The 'Golden Window':** Success in previous campaigns is the strongest predictor of future success, particularly if re-engaged within 180 days.
*   **Recommendation:** Prioritize cellular outreach for customers who converted in the past year.